In [2]:
!pip install torch

  Using cached torch-2.9.0-cp312-cp312-win_amd64.whl.metadata (30 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached torch-2.9.0-cp312-cp312-win_amd64.whl (109.3 MB)
Using cached fsspec-2025.10.0-py3-none-any.whl (200 kB)
Using cached networkx-3.5-py3-none-any.whl (2.0 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached filelock-3.20.0-py3-none-any.whl (16 kB)

   ---------------------------------------- 0/6 [mpmath]
   ---------------------------------------- 0/6 [mpmath]
   ---------------------------------------- 0/6 [mpmath]
   ---------------------------------------- 0/6 [mpmath]
   ---------------------------------------- 0/6 [mpma

## 1. Parameter tunning

In [13]:
import csv
import random
from Solver import Solver
from FieldClass import Field, build_mapping
from model.CNN_model import CNN_Network
from utils import generateField
import torch

# Tham số
max_neiborghs_list = [300, 350, 400, 450, 500]
T_starts = [1, 2, 3]
sizes = [24]
num_samples = 3

# CSV output
output_file = "sa_parameter_tuning_results_avg.csv"

# Mở file CSV để ghi kết quả
with open(output_file, mode='w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Size", "max_neiborghs", "T_start", "avg_score", "avg_steps"])

    for size in sizes:
        n_values = size * size // 2
        print(f"=== Size {size} ===")

        for maxN in max_neiborghs_list:
            for T in T_starts:
                total_score = 0
                total_steps = 0

                for sample_idx in range(num_samples):
                    # Sinh field ngẫu nhiên
                    entities = generateField(n=size)
                    mappings = build_mapping(n=size)
                    field = Field(size=size, entities=entities, mappings=mappings)

                    solver = Solver(init_field=field, max_depth=100)

                    best_field, solution = solver.simulated_annealing(
                        max_neiborghs=maxN,
                        T_start=T,
                        T_min=T*0.01,
                        alpha=0.995
                    )

                    score = best_field.score()
                    steps_count = len(solution)

                    total_score += score
                    total_steps += steps_count

                avg_score = total_score / num_samples
                avg_steps = total_steps / num_samples

                writer.writerow([size, maxN, T, avg_score, avg_steps])
                print(f"size={size}, maxN={maxN}, T={T}, avg_score={avg_score:.2f}, avg_steps={avg_steps:.1f}")


=== Size 24 ===
size=24, maxN=300, T=1, avg_score=260.33, avg_steps=8106.3
size=24, maxN=300, T=2, avg_score=256.33, avg_steps=8012.3
size=24, maxN=300, T=3, avg_score=256.00, avg_steps=8518.0
size=24, maxN=350, T=1, avg_score=264.33, avg_steps=9067.3
size=24, maxN=350, T=2, avg_score=260.67, avg_steps=9312.7
size=24, maxN=350, T=3, avg_score=261.67, avg_steps=9889.0
size=24, maxN=400, T=1, avg_score=271.67, avg_steps=10342.3
size=24, maxN=400, T=2, avg_score=269.00, avg_steps=10689.3
size=24, maxN=400, T=3, avg_score=269.00, avg_steps=11035.3
size=24, maxN=450, T=1, avg_score=273.00, avg_steps=11597.7
size=24, maxN=450, T=2, avg_score=267.00, avg_steps=11160.3
size=24, maxN=450, T=3, avg_score=267.67, avg_steps=11928.3
size=24, maxN=500, T=1, avg_score=274.67, avg_steps=13004.7
size=24, maxN=500, T=2, avg_score=274.00, avg_steps=13022.7
size=24, maxN=500, T=3, avg_score=274.33, avg_steps=13346.0


## 2. Tối ưu size

In [2]:
from FieldClass import Field, build_mapping
from utils import generateField
from Solver import Solver   
from postProcessing import removeDuplicate, removeSameState

n = 24
n_values = n * n // 2

entities = generateField(n = n)
mappings = build_mapping(n = n)
field = Field(size = n, entities = entities, mappings = mappings)

solver = Solver(init_field=field)
best_field, path = solver.simulated_annealing(max_neiborghs=500)

print(f"Best_score: {best_field.score()}")
print(f"Actions: {len(path)}")


Best_score: 281
Actions: 56064
